Model selection and cross-validation for soil moisture mapping
---------------------------------------------------------------

This notebook trains multiple models for soil moisture prediction maps. 
The model training data is based on weekly averaged soil moisture probes and multiple spatial-temporal dependent covariates. 

The prediction models that are tested are based on Gaussian Process regression with two different base function: Random Forest (RF) and Bayesian Linear Regression (BLR). The base functions are used to account for the non-linear relationship between the covariates and the soil moisture. Gaussian Process Regression (GPR) is used to account for the spatial-temporal correlation of the soil moisture probes. When GPR is enabled, base model were used as the mean function of GPR.

The model selection is based on the cross-validation of the prediction error (normalized RMSE, R2) for the test data.
Test data is selected based on n-fold cross-validation. Different test data is selected for each fold.

Note that test data is n-fold split based on the spatial location of the probes, and not on time. If test data would be selected based on unique location in space and time rather than on space only, the cross-validation would reveal in superficial low prediction errors, because the temporal variability of the soil moisture over short time periods is very low relative to the spatial variations, and hence test data would be strongly correlated with the training data.

User settings, such as input/output paths and all other options, are set in the settings file, e.g.:

`settings_testmodel_spatial_cal.yaml`

This package is part of the machine learning project developed for the Agricultural Research Federation (AgReFed).

## Library imports

In [1]:
import numpy as np
import pandas as pd
import os
import sys
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split 
import yaml
import argparse
from types import SimpleNamespace  
from matplotlib.image import imread
import time

# Custom local libraries:
sys.path.append('../../python_scripts')

from utils import print2, truncate_data
from preprocessing import gen_kfold
import GPmodel as gp # GP model plus kernel functions and distance matrix calculation
from soilmod_xval_cal import runmodel


 ## Reading and process settings

 All settings are specified in the .yaml file to make analysis reproducible. Below we will read and inspect the settings.

In [2]:
# Define name of settings file to save configuration
fname_settings = 'settings_testmodel_spatial_cal.yaml'
path_settings = '.'

In [3]:
# Load settings from yaml file
with open(os.path.join(path_settings,fname_settings), 'r') as f:
    settings = yaml.load(f, Loader=yaml.FullLoader)
# Parse settings dictinary as namespace (settings are available as 
# settings.variable_name rather than settings['variable_name'])
settings = SimpleNamespace(**settings)


# Add temporal or vertical component
if settings.axistype == 'temporal':
    settings.colname_zcoord = settings.colname_tcoord
    settings.colname_zmin = settings.colname_tmin
    settings.colname_zmax =  settings.colname_tmax

if type(settings.model_functions) != list:
    settings.model_functions = [settings.model_functions]

# check if outpath exists, if not create direcory
os.makedirs(settings.outpath, exist_ok = True)

# Intialise output info file:
print('init')
print(f'--- Parameter Settings ---')
print(f'Selected Model Functions: {settings.model_functions}')
print(f'Target Name: {settings.name_target}')
print(f'--------------------------')

# Print features selected
print("")
print("--- Features Selected ---")
for key in settings.__dict__:
    if key == "name_features":
        for feature in settings.name_features:
            print(f"'{feature}'")

init
--- Parameter Settings ---
Selected Model Functions: ['rf', 'xgb']
Target Name: SM
--------------------------

--- Features Selected ---
'NDVI'
'SND'
'CLY'
'L15'
'DepthBot'
'BDW'
'SOC'
'DepthTop'
'PAWC'
'Rain_df_950'
'Solar'
'ET_df_990'


## Data Preprocessing

The data preprocessing includes the following steps:
 - reading in the dataset from .csv file into pandas dataframe
 - checking coordinate names and converting if necessary to default name convention (x,y,z)
 - select data for top soil only
 - generating n-fold cross-validation test sets
 - converting coordinates to origin at x,y = 0,0

In [4]:
print('Reading data into dataframe...')
# Read in data
dfsel = pd.read_csv(os.path.join(settings.inpath, settings.infname))

# Rename x and y coordinates of input data
if settings.colname_xcoord != 'x':
    dfsel.rename(columns={settings.colname_xcoord: 'x'}, inplace = True)
if settings.colname_ycoord != 'y':
    dfsel.rename(columns={settings.colname_ycoord: 'y'}, inplace = True)
if (settings.axistype == 'vertical') & (settings.colname_zcoord != 'z'):
    dfsel.rename(columns={settings.colname_zcoord: 'z'}, inplace = True)
else:
    dfsel.rename(columns={settings.colname_tcoord: 'z'}, inplace = True)
    dfsel.rename(columns={settings.colname_zcoord: 'z'}, inplace = True)
settings.name_features.append('z')

# Select data between zmin and zmax
dfsel = dfsel[(dfsel['z'] >= settings.colname_zmin) & (dfsel['z'] <= settings.colname_zmax)]

# Generate n-fold indices
print(f'Generating {settings.nfold}-fold test sets based on moisture probe locations..')
# dfsel = gen_kfold(dfsel, nfold = settings.nfold, label_nfold = 'nfold', id_unique = ['z'], precision_unique = 0.01, sort=True)
dfsel = gen_kfold(dfsel, nfold = settings.nfold, label_nfold = 'nfold', id_unique = ['x', 'y'], precision_unique = 0.01, sort=False)

## Get coordinates for training data and set coord origin to (0,0)
print(f'Setting coordinate origin to (0,0)...')
bound_xmin = dfsel.x.min()
bound_xmax = dfsel.x.max()
bound_ymin = dfsel.y.min()
bound_ymax = dfsel.y.max()

# Set origin to (0,0)
dfsel['x'] = dfsel['x'] - bound_xmin
dfsel['y'] = dfsel['y'] - bound_ymin

print('Preprocessing data finished.')



Reading data into dataframe...
Generating 1-fold test sets based on moisture probe locations..
Setting coordinate origin to (0,0)...
Preprocessing data finished.


## Hyperparameter tuning

In [5]:
# Define stats result lists
stats_summaries = []

# Loop over model functions and evaluate
for model_function in settings.model_functions:
    # run and evaluate model
    time_start = time.time()
    dfsum, stats_summary, model_outpath = runmodel(dfsel, model_function, settings)
    time_end = time.time()
    print(f'Time elapsed: {(time_end - time_start)/3600} hours.)')
    print(f'All output files of {model_function} saved in {model_outpath}')
    print('')
    # save results
    stats_summaries.append(stats_summary)


Computing 1-fold xrossvalidation for function model: rf
Processing for nfold  1
Starting Grid Search CV for Random Forest
Fitting 5 folds for each of 108 candidates, totalling 540 fits
[CV 1/5] END max_depth=None, max_features=0.3, min_samples_leaf=1, min_samples_split=2, n_estimators=100;, score=(train=-0.000, test=-0.001) total time=   4.2s
[CV 2/5] END max_depth=None, max_features=0.3, min_samples_leaf=1, min_samples_split=2, n_estimators=100;, score=(train=-0.000, test=-0.001) total time=   4.2s
[CV 3/5] END max_depth=None, max_features=0.3, min_samples_leaf=1, min_samples_split=2, n_estimators=100;, score=(train=-0.000, test=-0.001) total time=   4.2s
[CV 4/5] END max_depth=None, max_features=0.3, min_samples_leaf=1, min_samples_split=2, n_estimators=100;, score=(train=-0.000, test=-0.001) total time=   4.2s
[CV 5/5] END max_depth=None, max_features=0.3, min_samples_leaf=1, min_samples_split=2, n_estimators=100;, score=(train=-0.000, test=-0.004) total time=   4.2s
[CV 1/5] END ma

ValueError: Found array with 0 sample(s) (shape=(0, 13)) while a minimum of 1 is required by RandomForestRegressor.

### Result
Best Parameters {'max_depth': None, 'max_features': 0.3, 'min_samples_leaf': 1, 'min_samples_split': 10, 'n_estimators': 100}


In [6]:
from datetime import datetime
datetime.now().strftime("%Y-%m-%d %H:%M")

'2025-06-03 12:50'